## SRP225204

**paper:** [PMID: 36001538](https://pmc.ncbi.nlm.nih.gov/articles/PMC9401185/) - Transcriptome profiling of blood from common bottlenose dolphins (Tursiops truncatus) in the northern Gulf of Mexico to enhance health assessment capabilities, 2022

**date, curator:** 2026-04-10, Sara Carsanaro

**notes**
* supplementary table 1 has age class and pregnancy status, which i added to the annotation file
    * not exactly sure how to annotated subadult stage, as many of these are pregnant so clearly they are sexually mature

### annotation summary

In [21]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,blood,UBERON:0000178,blood,perfect match


In [22]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,Adult,UBERON:0000113,post-juvenile adult stage,perfect match
2,Subadult,UBERON:0000113,post-juvenile adult stage,missing child term
13,Calf,UBERON:0000112,sexually immature stage,perfect match


### set variables, import packages, define functions

In [1]:
experiment_id = "SRP225204"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

## validation
path_to_v_script = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/validate_annotations.py'
path_to_rules = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/rules/'
val_output = "{}{}/validation.tsv".format(path_to_output_main, experiment_id)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
0it [00:00, ?it/s]
0 samples dont have attributes, try to find them somewhere else
0it [00:00, ?it/s]
0 samples dont have attributes


### library annnotations

In [4]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX6980739,SRP225204,Illumina NovaSeq 6000,SRS5505698,,,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4118986,blood,Adult,,,,F,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,YZ1.18B,"SAMN13019213,GSM4118986",,,,,,,,,,,10/04/2026,Whole blood RNA was extracted using a PAXgene Blood RNA Kit (Qiagen). Libraries were constructed using a NEBNext ultra Directional RNA Library Prep kit for Illumina and indexed with the NEBNextMultiplex Oligos for Illumina,,,,blood,,,,TRANSCRIPTOMIC,cDNA
1,SRX6980738,SRP225204,Illumina NovaSeq 6000,SRS5505697,,,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4118985,blood,Adult,,,,F,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,YY9.18B,"SAMN13019214,GSM4118985",,,,,,,,pregnant,,,10/04/2026,Whole blood RNA was extracted using a PAXgene Blood RNA Kit (Qiagen). Libraries were constructed using a NEBNext ultra Directional RNA Library Prep kit for Illumina and indexed with the NEBNextMultiplex Oligos for Illumina,,,,blood,,,,TRANSCRIPTOMIC,cDNA
2,SRX6980737,SRP225204,Illumina NovaSeq 6000,SRS5505696,,,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4118984,blood,Subadult,,,,F,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,YY3.18B,"SAMN13019215,GSM4118984",,,,,,,,,,,10/04/2026,Whole blood RNA was extracted using a PAXgene Blood RNA Kit (Qiagen). Libraries were constructed using a NEBNext ultra Directional RNA Library Prep kit for Illumina and indexed with the NEBNextMultiplex Oligos for Illumina,,,,blood,,,,TRANSCRIPTOMIC,cDNA
3,SRX6980736,SRP225204,Illumina NovaSeq 6000,SRS5505695,,,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4118983,blood,Subadult,,,,F,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,YX9.18B,"SAMN13019216,GSM4118983",,,,,,,,pregnant,,,10/04/2026,Whole blood RNA was extracted using a PAXgene Blood RNA Kit (Qiagen). Libraries were constructed using a NEBNext ultra Directional RNA Library Prep kit for Illumina and indexed with the NEBNextMultiplex Oligos for Illumina,,,,blood,,,,TRANSCRIPTOMIC,cDNA
4,SRX6980735,SRP225204,Illumina NovaSeq 6000,SRS5505694,,,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4118982,blood,Adult,,,,F,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,YX7.18B,"SAMN13019217,GSM4118982",,,,,,,,pregnant,,,10/04/2026,Whole blood RNA was extracted using a PAXgene Blood RNA Kit (Qiagen). Libraries were constructed using a NEBNext ultra Directional RNA Library Prep kit for Illumina and indexed with the NEBNextMultiplex Oligos for Illumina,,,,blood,,,,TRANSCRIPTOMIC,cDNA
5,SRX6980734,SRP225204,Illumina NovaSeq 6000,SRS5505693,,,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4118981,blood,Adult,,,,F,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,YX5.917B,"SAMN13019218,GSM4118981",,,,,,,,pregnant,,,10/04/2026,Whole blood RNA was extracted using a PAXgene Blood RNA Kit (Qiagen). Libraries were constructed using a NEBNext ultra Directional RNA Library Prep kit for Illumina and indexed with the NEBNextMultiplex Oligos for Illumina,,,,blood,,,,TRANSCRIPTOMIC,cDNA
6,SRX6980733,SRP225204,Illumina NovaSeq 6000,SRS5505692,,,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4118980,blood,Subadult,,,,F,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,YX3.917B,"SAMN13019219,GSM4118980",,,,,,,,,,,10/04/2026,Whole blood RN

#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [5]:
unique_sorted(library, "infoOrgan")

['blood']


In [6]:

# all
library.loc[:,'anatId'] = 'UBERON:0000178'
library.loc[:,'anatName'] = 'blood'
# perfect match, missing child term, other
library.loc[:,'anatAnnotationStatus'] = 'perfect match'
# partial sampling, full sampling, not documented
library.loc[:,'anatBiologicalStatus'] = 'not documented'


# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX6980739,SRP225204,Illumina NovaSeq 6000,SRS5505698,UBERON:0000178,blood,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4118986,blood,Adult,perfect match,not documented,,F,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,YZ1.18B,"SAMN13019213,GSM4118986",,,,,,,,,,,10/04/2026,Whole blood RNA was extracted using a PAXgene Blood RNA Kit (Qiagen). Libraries were constructed using a NEBNext ultra Directional RNA Library Prep kit for Illumina and indexed with the NEBNextMultiplex Oligos for Illumina,,,,blood,,,,TRANSCRIPTOMIC,cDNA
1,SRX6980738,SRP225204,Illumina NovaSeq 6000,SRS5505697,UBERON:0000178,blood,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4118985,blood,Adult,perfect match,not documented,,F,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,YY9.18B,"SAMN13019214,GSM4118985",,,,,,,,pregnant,,,10/04/2026,Whole blood RNA was extracted using a PAXgene Blood RNA Kit (Qiagen). Libraries were constructed using a NEBNext ultra Directional RNA Library Prep kit for Illumina and indexed with the NEBNextMultiplex Oligos for Illumina,,,,blood,,,,TRANSCRIPTOMIC,cDNA
2,SRX6980737,SRP225204,Illumina NovaSeq 6000,SRS5505696,UBERON:0000178,blood,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4118984,blood,Subadult,perfect match,not documented,,F,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,YY3.18B,"SAMN13019215,GSM4118984",,,,,,,,,,,10/04/2026,Whole blood RNA was extracted using a PAXgene Blood RNA Kit (Qiagen). Libraries were constructed using a NEBNext ultra Directional RNA Library Prep kit for Illumina and indexed with the NEBNextMultiplex Oligos for Illumina,,,,blood,,,,TRANSCRIPTOMIC,cDNA
3,SRX6980736,SRP225204,Illumina NovaSeq 6000,SRS5505695,UBERON:0000178,blood,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4118983,blood,Subadult,perfect match,not documented,,F,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,YX9.18B,"SAMN13019216,GSM4118983",,,,,,,,pregnant,,,10/04/2026,Whole blood RNA was extracted using a PAXgene Blood RNA Kit (Qiagen). Libraries were constructed using a NEBNext ultra Directional RNA Library Prep kit for Illumina and indexed with the NEBNextMultiplex Oligos for Illumina,,,,blood,,,,TRANSCRIPTOMIC,cDNA
4,SRX6980735,SRP225204,Illumina NovaSeq 6000,SRS5505694,UBERON:0000178,blood,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4118982,blood,Adult,perfect match,not documented,,F,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,YX7.18B,"SAMN13019217,GSM4118982",,,,,,,,pregnant,,,10/04/2026,Whole blood RNA was extracted using a PAXgene Blood RNA Kit (Qiagen). Libraries were constructed using a NEBNext ultra Directional RNA Library Prep kit for Illumina and indexed with the NEBNextMultiplex Oligos for Illumina,,,,blood,,,,TRANSCRIPTOMIC,cDNA
5,SRX6980734,SRP225204,Illumina NovaSeq 6000,SRS5505693,UBERON:0000178,blood,,,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4118981,blood,Adult,perfect match,not documented,,F,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,YX5.917B,"SAMN13019218,GSM4118981",,,,,,,,pregnant,,,10/04/2026,Whole blood RNA was extracted using a PAXgene Blood RNA Kit (Qiagen). Libraries were constructed using a NEBNext ultra Directional RNA Library Prep kit for Illumina and indexed with the NEBNextMultiplex Oligos for Illumina,,,,blood,,,,TRANSCRIPTOMIC,cDNA
6,SRX698

#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [7]:
unique_sorted(library, "infoStage")

['Adult' 'Calf' 'Subadult']


In [8]:
# Adult
library.loc[library["infoStage"] == "Adult", "stageId"] = "UBERON:0000113"
library.loc[library["infoStage"] == "Adult", "stageName"] = "post-juvenile adult stage"
# perfect match, missing child term, other
library.loc[library["infoStage"] == "Adult", "stageAnnotationStatus"] = "perfect match"

# Calf
library.loc[library["infoStage"] == "Calf", "stageId"] = "UBERON:0000112"
library.loc[library["infoStage"] == "Calf", "stageName"] = "sexually immature stage"
# perfect match, missing child term, other
library.loc[library["infoStage"] == "Calf", "stageAnnotationStatus"] = "perfect match"

# Subadult
library.loc[library["infoStage"] == "Subadult", "stageId"] = "UBERON:0000113"
library.loc[library["infoStage"] == "Subadult", "stageName"] = "post-juvenile adult stage"
# perfect match, missing child term, other
library.loc[library["infoStage"] == "Subadult", "stageAnnotationStatus"] = "missing child term"

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX6980739,SRP225204,Illumina NovaSeq 6000,SRS5505698,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4118986,blood,Adult,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,YZ1.18B,"SAMN13019213,GSM4118986",,,,,,,,,,,10/04/2026,Whole blood RNA was extracted using a PAXgene Blood RNA Kit (Qiagen). Libraries were constructed using a NEBNext ultra Directional RNA Library Prep kit for Illumina and indexed with the NEBNextMultiplex Oligos for Illumina,,,,blood,,,,TRANSCRIPTOMIC,cDNA
1,SRX6980738,SRP225204,Illumina NovaSeq 6000,SRS5505697,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4118985,blood,Adult,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,YY9.18B,"SAMN13019214,GSM4118985",,,,,,,,pregnant,,,10/04/2026,Whole blood RNA was extracted using a PAXgene Blood RNA Kit (Qiagen). Libraries were constructed using a NEBNext ultra Directional RNA Library Prep kit for Illumina and indexed with the NEBNextMultiplex Oligos for Illumina,,,,blood,,,,TRANSCRIPTOMIC,cDNA
2,SRX6980737,SRP225204,Illumina NovaSeq 6000,SRS5505696,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4118984,blood,Subadult,perfect match,not documented,missing child term,F,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,YY3.18B,"SAMN13019215,GSM4118984",,,,,,,,,,,10/04/2026,Whole blood RNA was extracted using a PAXgene Blood RNA Kit (Qiagen). Libraries were constructed using a NEBNext ultra Directional RNA Library Prep kit for Illumina and indexed with the NEBNextMultiplex Oligos for Illumina,,,,blood,,,,TRANSCRIPTOMIC,cDNA
3,SRX6980736,SRP225204,Illumina NovaSeq 6000,SRS5505695,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4118983,blood,Subadult,perfect match,not documented,missing child term,F,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,YX9.18B,"SAMN13019216,GSM4118983",,,,,,,,pregnant,,,10/04/2026,Whole blood RNA was extracted using a PAXgene Blood RNA Kit (Qiagen). Libraries were constructed using a NEBNext ultra Directional RNA Library Prep kit for Illumina and indexed with the NEBNextMultiplex Oligos for Illumina,,,,blood,,,,TRANSCRIPTOMIC,cDNA
4,SRX6980735,SRP225204,Illumina NovaSeq 6000,SRS5505694,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4118982,blood,Adult,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,YX7.18B,"SAMN13019217,GSM4118982",,,,,,,,pregnant,,,10/04/2026,Whole blood RNA was extracted using a PAXgene Blood RNA Kit (Qiagen). Libraries were constructed using a NEBNext ultra Directional RNA Library Prep kit for Illumina and indexed with the NEBNextMultiplex Oligos for Illumina,,,,blood,,,,TRANSCRIPTOMIC,cDNA
5,SRX6980734,SRP225204,Illumina NovaSeq 6000,SRS5505693,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4118981,blood,Adult,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,nan,,,YX5.9

#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [ ]:
#library.loc[:,'sex'] = ''
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [9]:
# making these variables because we use them again in the experiment file
my_protocol = 'NEBNext Ultra Directional RNA Library Prep Kit'
# full_length or 3'
my_protocolType = 'full_length'

library.loc[:,'protocol'] = my_protocol
library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'polyA'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX6980739,SRP225204,Illumina NovaSeq 6000,SRS5505698,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4118986,blood,Adult,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,YZ1.18B,"SAMN13019213,GSM4118986",,,,,,,,,,,10/04/2026,Whole blood RNA was extracted using a PAXgene Blood RNA Kit (Qiagen). Libraries were constructed using a NEBNext ultra Directional RNA Library Prep kit for Illumina and indexed with the NEBNextMultiplex Oligos for Illumina,,,,blood,,,,TRANSCRIPTOMIC,cDNA
1,SRX6980738,SRP225204,Illumina NovaSeq 6000,SRS5505697,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4118985,blood,Adult,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,YY9.18B,"SAMN13019214,GSM4118985",,,,,,,,pregnant,,,10/04/2026,Whole blood RNA was extracted using a PAXgene Blood RNA Kit (Qiagen). Libraries were constructed using a NEBNext ultra Directional RNA Library Prep kit for Illumina and indexed with the NEBNextMultiplex Oligos for Illumina,,,,blood,,,,TRANSCRIPTOMIC,cDNA
2,SRX6980737,SRP225204,Illumina NovaSeq 6000,SRS5505696,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4118984,blood,Subadult,perfect match,not documented,missing child term,F,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,YY3.18B,"SAMN13019215,GSM4118984",,,,,,,,,,,10/04/2026,Whole blood RNA was extracted using a PAXgene Blood RNA Kit (Qiagen). Libraries were constructed using a NEBNext ultra Directional RNA Library Prep kit for Illumina and indexed with the NEBNextMultiplex Oligos for Illumina,,,,blood,,,,TRANSCRIPTOMIC,cDNA
3,SRX6980736,SRP225204,Illumina NovaSeq 6000,SRS5505695,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4118983,blood,Subadult,perfect match,not documented,missing child term,F,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,YX9.18B,"SAMN13019216,GSM4118983",,,,,,,,pregnant,,,10/04/2026,Whole blood RNA was extracted using a PAXgene Blood RNA Kit (Qiagen). Libraries were constructed using a NEBNext ultra Directional RNA Library Prep kit for Illumina and indexed with the NEBNextMultiplex Oligos for Illumina,,,,blood,,,,TRANSCRIPTOMIC,cDNA
4,SRX6980735,SRP225204,Illumina NovaSeq 6000,SRS5505694,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4118982,blood,Adult,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,YX7.18B,"SAMN13019217,GSM4118982",,,,,,,,pregnant,,,10/04/2026,Whole blood RNA was extracted using a PAXgene Blood RNA Kit (Qiagen). Libraries were constructed using a NEBNext ultra Directional RNA Library Prep kit for Illumina and indexed with the NEBNextMultiplex Oligos for Illumina,,,,blood,,,,TRANSCRIPTOMIC,cDNA
5,SRX6980734,SRP225204,Illumina NovaSeq 6000,SRS5505693,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4118981,blood,Adult,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,p

#### globin, replicates

In [10]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

no duplicate values in SRSId


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [ ]:
#library.loc[:,'sampleAge_value'] = ''
#library.loc[:,'sampleAge_unit'] = ''

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [11]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-04-10'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX6980739,SRP225204,Illumina NovaSeq 6000,SRS5505698,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4118986,blood,Adult,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,YZ1.18B,"SAMN13019213,GSM4118986",,,,,,,,,,SAC,2026-04-10,Whole blood RNA was extracted using a PAXgene Blood RNA Kit (Qiagen). Libraries were constructed using a NEBNext ultra Directional RNA Library Prep kit for Illumina and indexed with the NEBNextMultiplex Oligos for Illumina,,,,blood,,,,TRANSCRIPTOMIC,cDNA
1,SRX6980738,SRP225204,Illumina NovaSeq 6000,SRS5505697,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4118985,blood,Adult,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,YY9.18B,"SAMN13019214,GSM4118985",,,,,,,,pregnant,,SAC,2026-04-10,Whole blood RNA was extracted using a PAXgene Blood RNA Kit (Qiagen). Libraries were constructed using a NEBNext ultra Directional RNA Library Prep kit for Illumina and indexed with the NEBNextMultiplex Oligos for Illumina,,,,blood,,,,TRANSCRIPTOMIC,cDNA
2,SRX6980737,SRP225204,Illumina NovaSeq 6000,SRS5505696,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4118984,blood,Subadult,perfect match,not documented,missing child term,F,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,YY3.18B,"SAMN13019215,GSM4118984",,,,,,,,,,SAC,2026-04-10,Whole blood RNA was extracted using a PAXgene Blood RNA Kit (Qiagen). Libraries were constructed using a NEBNext ultra Directional RNA Library Prep kit for Illumina and indexed with the NEBNextMultiplex Oligos for Illumina,,,,blood,,,,TRANSCRIPTOMIC,cDNA
3,SRX6980736,SRP225204,Illumina NovaSeq 6000,SRS5505695,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4118983,blood,Subadult,perfect match,not documented,missing child term,F,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,YX9.18B,"SAMN13019216,GSM4118983",,,,,,,,pregnant,,SAC,2026-04-10,Whole blood RNA was extracted using a PAXgene Blood RNA Kit (Qiagen). Libraries were constructed using a NEBNext ultra Directional RNA Library Prep kit for Illumina and indexed with the NEBNextMultiplex Oligos for Illumina,,,,blood,,,,TRANSCRIPTOMIC,cDNA
4,SRX6980735,SRP225204,Illumina NovaSeq 6000,SRS5505694,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4118982,blood,Adult,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,YX7.18B,"SAMN13019217,GSM4118982",,,,,,,,pregnant,,SAC,2026-04-10,Whole blood RNA was extracted using a PAXgene Blood RNA Kit (Qiagen). Libraries were constructed using a NEBNext ultra Directional RNA Library Prep kit for Illumina and indexed with the NEBNextMultiplex Oligos for Illumina,,,,blood,,,,TRANSCRIPTOMIC,cDNA
5,SRX6980734,SRP225204,Illumina NovaSeq 6000,SRS5505693,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4118981,blood,Adult,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra Directional RNA Library Prep Ki

#### comments

In [12]:
library.loc[:,'comment'] = 'PMID: 36001538, the adult, subadult and calf classifications are a bit confusing, especially because multiple subadult animals are pregnant thus cannot be sexually immature stage'

#### save complete file with correct columns

In [13]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX6980739,SRP225204,Illumina NovaSeq 6000,SRS5505698,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4118986,blood,Adult,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,YZ1.18B,"SAMN13019213,GSM4118986",,,,,,,"PMID: 36001538, the adult, subadult and calf classifications are a bit confusing, especially because multiple subadult animals are pregnant thus cannot be sexually immature stage",,,SAC,2026-04-10
1,SRX6980738,SRP225204,Illumina NovaSeq 6000,SRS5505697,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4118985,blood,Adult,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,YY9.18B,"SAMN13019214,GSM4118985",,,,,,,"PMID: 36001538, the adult, subadult and calf classifications are a bit confusing, especially because multiple subadult animals are pregnant thus cannot be sexually immature stage",pregnant,,SAC,2026-04-10
2,SRX6980737,SRP225204,Illumina NovaSeq 6000,SRS5505696,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4118984,blood,Subadult,perfect match,not documented,missing child term,F,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,YY3.18B,"SAMN13019215,GSM4118984",,,,,,,"PMID: 36001538, the adult, subadult and calf classifications are a bit confusing, especially because multiple subadult animals are pregnant thus cannot be sexually immature stage",,,SAC,2026-04-10
3,SRX6980736,SRP225204,Illumina NovaSeq 6000,SRS5505695,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4118983,blood,Subadult,perfect match,not documented,missing child term,F,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,YX9.18B,"SAMN13019216,GSM4118983",,,,,,,"PMID: 36001538, the adult, subadult and calf classifications are a bit confusing, especially because multiple subadult animals are pregnant thus cannot be sexually immature stage",pregnant,,SAC,2026-04-10
4,SRX6980735,SRP225204,Illumina NovaSeq 6000,SRS5505694,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4118982,blood,Adult,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,YX7.18B,"SAMN13019217,GSM4118982",,,,,,,"PMID: 36001538, the adult, subadult and calf classifications are a bit confusing, especially because multiple subadult animals are pregnant thus cannot be sexually immature stage",pregnant,,SAC,2026-04-10
5,SRX6980734,SRP225204,Illumina NovaSeq 6000,SRS5505693,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4118981,blood,Adult,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,YX5.917B,"SAMN13019218,GSM4118981",,,,,,,"PMID: 36001538, the adult, subadult and calf classifications are a bit confusing, especially because multiple subadult animals are pregnant thus cannot be sexually immature stage",pregnant,,SAC,2026-04-10
6,SRX6980733,SRP225204,Illumina NovaSeq 6000,SRS5505692,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4118980,blood,Subadult,perfect match,not documented,missing child term,F,,,

### experiment annotations

In [14]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP225204,RNA-seq analysis of archived blood from common bottlenose dolphin health assessments in the northern Gulf of Mexico,"Archived blood samples collected during common bottlenose dolphin health assessments in the northern Gulf of Mexico from 2013 to 2018 were analyzed by RNA-seq to support and enhance the assessment of animal health. The transcriptomic data were analyzed in conjunction with the substantial pool of health and environmental data collected during health assessments to investigate the utility of transcriptomic data in overall assessment of dolphin health and/or as markers of specific health concerns. Overall design: Illumina based RNA-seq analyses were carried out on RNA extracts of whole blood from 76 archived samples using the Tursiops truncatus NIST Tur_tru v1 genome as a guide. Samples were collected in Barataria Bay, LA (n=60) or Sarasota Bay, FL (n=16) from 2013-2018. Samples from Sarasota Bay served as a reference population that had not been exposed to substantial oil, whereas the samples from Barataria Bay come from a population directly impacted by the Deepwater Horizon oil spill.",SRA,,,,NEBNext Ultra Directional RNA Library Prep Kit,full_length,GSE138765,PRJNA577100,36001538,,10.1371/journal.pone.0272345,,


#### experiment and protocol details

In [15]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

76

In [16]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'total'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
experiment.loc[:,'protocol'] = my_protocol
experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP225204,RNA-seq analysis of archived blood from common bottlenose dolphin health assessments in the northern Gulf of Mexico,"Archived blood samples collected during common bottlenose dolphin health assessments in the northern Gulf of Mexico from 2013 to 2018 were analyzed by RNA-seq to support and enhance the assessment of animal health. The transcriptomic data were analyzed in conjunction with the substantial pool of health and environmental data collected during health assessments to investigate the utility of transcriptomic data in overall assessment of dolphin health and/or as markers of specific health concerns. Overall design: Illumina based RNA-seq analyses were carried out on RNA extracts of whole blood from 76 archived samples using the Tursiops truncatus NIST Tur_tru v1 genome as a guide. Samples were collected in Barataria Bay, LA (n=60) or Sarasota Bay, FL (n=16) from 2013-2018. Samples from Sarasota Bay served as a reference population that had not been exposed to substantial oil, whereas the samples from Barataria Bay come from a population directly impacted by the Deepwater Horizon oil spill.",SRA,total,Bgee 1K,76,NEBNext Ultra Directional RNA Library Prep Kit,full_length,GSE138765,PRJNA577100,36001538,,10.1371/journal.pone.0272345,,


#### paper and xrefs

In [17]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
experiment.loc[:,'PMID'] = '36001538'
experiment.loc[:,'reference_url'] = 'https://pmc.ncbi.nlm.nih.gov/articles/PMC9401185/'
experiment.loc[:,'DOI'] = '10.1371/journal.pone.0272345'
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP225204,RNA-seq analysis of archived blood from common bottlenose dolphin health assessments in the northern Gulf of Mexico,"Archived blood samples collected during common bottlenose dolphin health assessments in the northern Gulf of Mexico from 2013 to 2018 were analyzed by RNA-seq to support and enhance the assessment of animal health. The transcriptomic data were analyzed in conjunction with the substantial pool of health and environmental data collected during health assessments to investigate the utility of transcriptomic data in overall assessment of dolphin health and/or as markers of specific health concerns. Overall design: Illumina based RNA-seq analyses were carried out on RNA extracts of whole blood from 76 archived samples using the Tursiops truncatus NIST Tur_tru v1 genome as a guide. Samples were collected in Barataria Bay, LA (n=60) or Sarasota Bay, FL (n=16) from 2013-2018. Samples from Sarasota Bay served as a reference population that had not been exposed to substantial oil, whereas the samples from Barataria Bay come from a population directly impacted by the Deepwater Horizon oil spill.",SRA,total,Bgee 1K,76,NEBNext Ultra Directional RNA Library Prep Kit,full_length,GSE138765,PRJNA577100,36001538,https://pmc.ncbi.nlm.nih.gov/articles/PMC9401185/,10.1371/journal.pone.0272345,,


#### comments

In [ ]:
#experiment.loc[:,'comment'] = ''

display_df(experiment)

#### save complete file

In [18]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [19]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

In [20]:
! python3 $path_to_v_script --bulk-exp $experiment_to_add_path --bulk-lib $library_to_add_path --rules-dir $path_to_rules --out $val_output --strict

Total issues: 0
Errors: 0
Warnings: 0
Top codes:


#### check columns match

In [23]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [24]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
61156,SRX2792717,SRP106690,Illumina HiSeq 2500,SRS2174075,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,skin,Adult,perfect match,not documented,perfect match,M,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,SAB_TYP150722-02_AB,SAMN06909708,,,,,,,no paper,,,SAC,2026-04-10
61157,SRX2792716,SRP106690,Illumina HiSeq 2500,SRS2174074,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,skin,Adult,perfect match,not documented,perfect match,M,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,SAB_TYP150722-01_AA,SAMN06909707,,,,,,,no paper,,,SAC,2026-04-10
61158,SRX6980739,SRP225204,Illumina NovaSeq 6000,SRS5505698,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,blood,Adult,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,YZ1.18B,"SAMN13019213,GSM4118986",,,,,,,"PMID: 36001538, the adult, subadult and calf c...",,,SAC,2026-04-10
61159,SRX6980738,SRP225204,Illumina NovaSeq 6000,SRS5505697,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,blood,Adult,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,YY9.18B,"SAMN13019214,GSM4118985",,,,,,,"PMID: 36001538, the adult, subadult and calf c...",pregnant,,SAC,2026-04-10
61160,SRX6980737,SRP225204,Illumina NovaSeq 6000,SRS5505696,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,blood,Subadult,perfect match,not documented,missing child term,F,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,YY3.18B,"SAMN13019215,GSM4118984",,,,,,,"PMID: 36001538, the adult, subadult and calf c...",,,SAC,2026-04-10
61161,SRX6980736,SRP225204,Illumina NovaSeq 6000,SRS5505695,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,blood,Subadult,perfect match,not documented,missing child term,F,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,YX9.18B,"SAMN13019216,GSM4118983",,,,,,,"PMID: 36001538, the adult, subadult and calf c...",pregnant,,SAC,2026-04-10
61162,SRX6980735,SRP225204,Illumina NovaSeq 6000,SRS5505694,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,blood,Adult,perfect match,not documented,perfect match,F,,,9739,NEBNext Ultra Directional RNA Library Prep Kit,full_length,polyA,,,YX7.18B,"SAMN13019217,GSM4118982",,,,,,,"PMID: 36001538, the adult, subadult and calf c...",pregnant,,SAC,2026-04-10


In [25]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1168,SRP094638,Skin Transcriptomics of Common Bottlenose Dolp...,Common bottlenose dolphins serve as sentinels ...,SRA,total,Bgee 1K,116,NEBNext Ultra Directional RNA Library Prep Kit,full_length,GSE90941,PRJNA356411,28843847,https://www.sciencedirect.com/science/article/...,10.1016/j.margen.2017.08.002,,
1169,SRP106690,Tursiops truncatus Raw sequence reads,"RNA-Seq from bottlenose dolphin skin, location...",SRA,total,Bgee 1K,35,NEBNext Ultra Directional RNA Library Prep Kit,full_length,,PRJNA385781,,,,,no paper for this dataset
1170,SRP225204,RNA-seq analysis of archived blood from common...,Archived blood samples collected during common...,SRA,total,Bgee 1K,76,NEBNext Ultra Directional RNA Library Prep Kit,full_length,GSE138765,PRJNA577100,36001538,https://pmc.ncbi.nlm.nih.gov/articles/PMC9401185/,10.1371/journal.pone.0272345,,


### add annotations to git

In [26]:
! git pull

Already up to date.


In [27]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [28]:
! git status

On branch develop
Your branch is up to date with 'origin/develop'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   ../../../RNA_Seq/RNASeqExperiment.tsv
	modified:   ../../../RNA_Seq/RNASeqLibrary.tsv

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	../1_reject_experiment/out/
	../NCBI_output/
	./

no changes added to commit (use "git add" and/or "git commit -a")


In [29]:
! git add $git_experiment_path $git_library_path

In [30]:
! git commit -m $commit_message_exp

[develop d723d03] adding annotated bulk experiment SRP225204
 2 files changed, 77 insertions(+)


In [31]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 4.56 KiB | 4.56 MiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git/
   6eac936..d723d03  develop -> develop


### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push